## Training the Final Chess Data with DecisionTree, RandomForest and XGBoost

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import randint

#### Loading the Data

In [3]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir,"data/final_chess.csv"), parse_dates=["Date"])

In [4]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3255656 entries, 0 to 3255655
Data columns (total 98 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Site                    str           
 1   Date                    datetime64[us]
 2   White                   str           
 3   Black                   str           
 4   Result                  float64       
 5   WhiteElo                float64       
 6   BlackElo                float64       
 7   ECO                     str           
 8   EloDiff                 float64       
 9   absEloDiff              float64       
 10  AvgElo                  float64       
 11  HigherRatedPlayer       int64         
 12  WhiteGames              int64         
 13  WhiteWin                int64         
 14  WhiteLoss               int64         
 15  WhiteDraw               int64         
 16  BlackGames              int64         
 17  BlackWin                int64         
 18  BlackLoss    

#### Preparing Data for Training

In [ ]:
X = games.drop(columns=[
    "White", "Black", "Result", "Site", "ECO",

    # White/Black color counts
    'WasBWinRate', 'WasBLossRate', 'WasBDrawRate', 'BasWWinRate', 'BasWLossRate', 'BasWDrawRate',
    "WasBGames", "WasBWin", "WasBLoss", "WasBDraw",
    "BasWGames", "BasWWin", "BasWLoss", "BasWDraw",
])
y = games["Result"].map({1: 2, 0.5: 1, 0:0}) #sklearn only considers integers 

"""
0 = Black win
1 = Draw
2 = White win
"""

'\n0 = Black win\n1 = Draw\n2 = White win\n'

In [6]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3255656 entries, 0 to 3255655
Data columns (total 79 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Date                    datetime64[us]
 1   WhiteElo                float64       
 2   BlackElo                float64       
 3   EloDiff                 float64       
 4   absEloDiff              float64       
 5   AvgElo                  float64       
 6   HigherRatedPlayer       int64         
 7   WhiteGames              int64         
 8   WhiteWin                int64         
 9   WhiteLoss               int64         
 10  WhiteDraw               int64         
 11  BlackGames              int64         
 12  BlackWin                int64         
 13  BlackLoss               int64         
 14  BlackDraw               int64         
 15  WhiteWinRate            float64       
 16  WhiteLossRate           float64       
 17  WhiteDrawRate           float64       
 18  BlackWinRate 

In [6]:
n = len(games)
s1 = int(0.7 * n)
s2 = int(0.85 * n)

In [7]:
X_train = X.iloc[:s1].drop(columns=["Date"])
y_train = y.iloc[:s1].drop(columns=["Date"])
X_val = X.iloc[s1:s2].drop(columns=["Date"])
y_val = y.iloc[s1:s2].drop(columns=["Date"])
X_test = X.iloc[s2:].drop(columns=["Date"])
y_test = y.iloc[s2:].drop(columns=["Date"])

### Decision Tree

In [9]:
from sklearn.tree import DecisionTreeClassifier

#### RandomizedSearch



In [21]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [51]:
indices = np.arange(0, n)
sample_indices = np.random.choice(indices, size=500000, replace=False)
X_srch = X.iloc[sample_indices]
y_srch = y.iloc[sample_indices]

In [26]:
tscv = TimeSeriesSplit(5)

In [27]:
param_grid = {
    "max_depth": [10, 20, 30, 40, 60, 80, 100, None],
    "min_samples_split": np.arange(2,500),
    "min_samples_leaf": np.arange(1,200),
    "max_features": [None, "sqrt", 0.7],
    "max_leaf_nodes": [None, 100, 250, 500, 750, 1000],
    "class_weight": [None, "balanced"],
    "ccp_alpha": [0.0, 1e-5, 1e-4, 1e-3]
}

In [40]:
random_search = RandomizedSearchCV(DecisionTreeClassifier(criterion="gini", random_state=67), param_grid, n_iter=100, scoring="accuracy", cv=tscv, n_jobs=4, random_state=67)

In [ ]:
random_search.fit(X_srch, y_srch)

/Users/vallurileelasaikrishna/Documents/chess/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=67)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'ccp_alpha': [0.0, 1e-05, ...], 'class_weight': [None, 'balanced'], 'max_depth': [10, 20, ...], 'max_features': [None, 'sqrt', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",4
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",67
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``

In [ ]:
random_search.best_score_

np.float64(0.5902055608222433)

In [ ]:
results = pd.DataFrame(random_search.cv_results_)
results.head()

#### On actual Data

In [9]:
s3 = int(0.3 * n)
s4 = int(0.75 * n)
s5 = int(0.9 * n)
X_ntrain = X.iloc[s3:s4]
y_ntrain = y.iloc[s3:s4]
X_nval = X.iloc[s4:s5]
y_nval = y.iloc[s4:s5]

In [7]:
Xtrain = X[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
ytrain = y[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2024)].drop(columns="Date")
Xval   = X[X["Date"].dt.year == 2025].drop(columns="Date")
yval   = y[X["Date"].dt.year == 2025].drop(columns="Date")
Xtest  = X[X["Date"].dt.year == 2026].drop(columns="Date")
ytest  = y[X["Date"].dt.year == 2026].drop(columns="Date")

In [10]:
#parameters from the randomized search

dtc = DecisionTreeClassifier(ccp_alpha=1e-05, class_weight={
    0: 1.0,
    1: 1.5,
    2: 1.0
}, max_depth=20,
                       max_leaf_nodes=1000, min_samples_leaf=168,
                       min_samples_split=654, random_state=67)

In [11]:
dtc.fit(Xtrain, ytrain)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",654
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",168
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",67
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",1000
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.","{0: 1.0, 1: 1.5, 2: 1.0}"
,"ccp_alpha ccp_alpha: non-negative float, default=0.0Complexity parameter used for Minimal Cost-Complexity Pruning. Thesubtree with the largest cost complexity that is smaller than``ccp_alpha`` will be chosen. By default, no pruning is performed. See:ref:`minimal_cost_complexity_pruning` for details. See:ref:`sphx_glr_auto_examples_tree_plot_cost_complexity_pruning.py`for an example of such pruning... versionadded:: 0.22",1e-05
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstr

In [12]:
y_pred = dtc.predict(Xval)

In [13]:
# accuracy
np.mean(y_pred==yval)

np.float64(0.6016069181889124)

In [14]:
confusion_matrix(yval, y_pred)

array([[ 86687,  18167,  33894],
       [ 19820,  30394,  23747],
       [ 34669,  19796, 109572]])

In [15]:
classification_report(yval, y_pred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.61      0.62      0.62    138748',
 '           1       0.44      0.41      0.43     73961',
 '           2       0.66      0.67      0.66    164037',
 '',
 '    accuracy                           0.60    376746',
 '   macro avg       0.57      0.57      0.57    376746',
 'weighted avg       0.60      0.60      0.60    376746']

In [39]:
pd.DataFrame(dtc.feature_importances_, index=Xtrain.columns).sort_values(by=0).tail(45)

,0
GamesDiff,0.000094
BasBWin,0.000110
BasBLoss,0.000117
h2h_Draw,0.000124
WLast10loss,0.000226
BasBDraw,0.000241
WhiteWinRate,0.000246
BlackDrawStreak,0.000288
WhiteLossStreak,0.000308
WhiteDraw,0.000391


saving the model in results

In [ ]:
#joblib.dump(dtc, os.path.join(project_dir, "results/models/dtc_60.03.joblib"))

['/Users/vallurileelasaikrishna/Documents/chess/results/models/dtc_60.03.joblib']

### Random Forest

#### Randomized search for rfc

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [52]:
tscv = TimeSeriesSplit(3)

In [53]:
param_grid = {
    "n_estimators" : randint(99, 699),
    "max_depth" : [20, 30, 40, 50, 60, None],
    "min_samples_split" : randint(250, 700),
    "min_samples_leaf" : randint(51, 400),
    "max_features" : [None, "sqrt", 0.7],
    "class_weight" : [None, "balanced"],
    "ccp_alpha" : [0, 1e-05]
}

In [54]:
random_search = RandomizedSearchCV(RandomForestClassifier(bootstrap=True, random_state=67, n_jobs=-1, oob_score=True),
                                   param_distributions=param_grid,
                                   n_iter=20,
                                   cv=tscv,
                                   random_state=67,
                                   n_jobs=-1,
                                   scoring="accuracy",
                                   return_train_score=True,
                                   verbose=2
                                   )

In [55]:
random_search.fit(X_srch, y_srch)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=None, max_features=sqrt, min_samples_leaf=349, min_samples_split=692, n_estimators=125; total time= 1.3min
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=None, max_features=sqrt, min_samples_leaf=349, min_samples_split=692, n_estimators=125; total time= 3.0min
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=None, max_features=sqrt, min_samples_leaf=349, min_samples_split=692, n_estimators=125; total time= 5.2min
[CV] END ccp_alpha=1e-05, class_weight=balanced, max_depth=40, max_features=sqrt, min_samples_leaf=58, min_samples_split=643, n_estimators=646; total time= 7.4min
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=60, max_features=0.7, min_samples_leaf=225, min_samples_split=675, n_estimators=249; total time=10.3min
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=None, max_features=None, min_samples_leaf=214, min_samples_split=344, n_esti

/Users/vallurileelasaikrishna/Documents/chess/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END ccp_alpha=1e-05, class_weight=None, max_depth=40, max_features=None, min_samples_leaf=381, min_samples_split=295, n_estimators=212; total time=12.5min
[CV] END ccp_alpha=1e-05, class_weight=balanced, max_depth=50, max_features=None, min_samples_leaf=344, min_samples_split=458, n_estimators=286; total time=41.5min
[CV] END ccp_alpha=0, class_weight=balanced, max_depth=30, max_features=0.7, min_samples_leaf=328, min_samples_split=343, n_estimators=197; total time=36.0min
[CV] END ccp_alpha=1e-05, class_weight=None, max_depth=20, max_features=0.7, min_samples_leaf=74, min_samples_split=406, n_estimators=121; total time=23.4min
[CV] END ccp_alpha=1e-05, class_weight=balanced, max_depth=20, max_features=0.7, min_samples_leaf=90, min_samples_split=540, n_estimators=581; total time=110.0min
[CV] END ccp_alpha=0, class_weight=None, max_depth=20, max_features=0.7, min_samples_leaf=295, min_samples_split=436, n_estimators=211; total time= 9.1min
[CV] END ccp_alpha=1e-05, class_weight=No

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=67)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'ccp_alpha': [0, 1e-05], 'class_weight': [None, 'balanced'], 'max_depth': [20, 30, ...], 'max_features': [None, 'sqrt', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",67
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to s

In [56]:
random_search.best_score_

np.float64(0.5995946666666667)

In [57]:
results = pd.DataFrame(random_search.cv_results_)
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_ccp_alpha,param_class_weight,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,...,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
0,1106.191842,567.446399,9.253906,0.539031,0.00001,balanced,40,sqrt,58,643,...,0.585832,0.585864,0.584717,0.001599,9,0.592984,0.594180,0.595101,0.594088,0.000867
1,2489.222211,1272.579373,4.711960,0.960052,0.00000,balanced,None,None,214,344,...,0.584368,0.584184,0.582387,0.002673,19,0.593688,0.594544,0.595901,0.594711,0.000911
2,188.027809,94.336632,1.563711,0.121787,0.00000,balanced,None,sqrt,349,692,...,0.584208,0.583632,0.582835,0.001553,15,0.587296,0.589096,0.589664,0.588685,0.001009
3,3977.119808,2077.499409,8.453291,0.897590,0.00001,balanced,20,0.7,90,540,...,0.585584,0.585080,0.583589,0.002473,12,0.595416,0.596460,0.597939,0.596605,0.001035
4,1567.548978,825.357511,3.296046,0.448132,0.00000,balanced,60,0.7,225,675,...,0.584104,0.584480,0.582467,0.002586,17,0.589976,0.590960,0.592008,0.590981,0.000830


#### On actual data

In [20]:
rfc = RandomForestClassifier(ccp_alpha=1e-05, max_depth=20, max_features=0.7,
                       min_samples_leaf=74, min_samples_split=406,
                       n_estimators=121, n_jobs=-1, oob_score=True,
                       random_state=67)

In [21]:
rfc.get_params()

{'bootstrap': True,
 'ccp_alpha': 1e-05,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 20,
 'max_features': 0.7,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 74,
 'min_samples_split': 406,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 121,
 'n_jobs': -1,
 'oob_score': True,
 'random_state': 67,
 'verbose': 0,
 'warm_start': False}

In [23]:
rfc.fit(Xtrain, ytrain)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",121
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",406
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",74
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.7
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",67
,"ccp_alpha ccp_alpha: non-negative float, default=0.0Complexity parameter used for Minimal Cost-Complexity Pruning. Thesubtree with the largest cost complexity that is smaller than``ccp_alpha`` will be chosen. By default, no pruning is performed. See:ref:`minimal_cost_complexity_pruning` for details. See:ref:`sphx_glr_auto_examples_tree_plot_cost_complexity_pruning.py`for an example of such pruning... versionadded:: 0.22",1e-05
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_m

In [24]:
rfc.oob_score_

0.5997278837375793

In [25]:
y_pred = rfc.predict(Xval)

In [26]:
np.mean(y_pred==yval) # accuracy

np.float64(0.6102599629458574)

In [27]:
confusion_matrix(yval, y_pred)

array([[ 93967,   7090,  37691],
       [ 26189,  16703,  31069],
       [ 37373,   7421, 119243]])

In [28]:
classification_report(yval, y_pred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.60      0.68      0.63    138748',
 '           1       0.54      0.23      0.32     73961',
 '           2       0.63      0.73      0.68    164037',
 '',
 '    accuracy                           0.61    376746',
 '   macro avg       0.59      0.54      0.54    376746',
 'weighted avg       0.60      0.61      0.59    376746']

In [31]:
pd.DataFrame(rfc.feature_importances_, index=X.columns).sort_values(by=0).tail(42)

ValueError: Shape of passed values is (78, 1), indices imply (79, 1)

saving the model

In [ ]:
#joblib.dump(rfc, os.path.join(project_dir, "results/models/rfc_61.18.joblib"))

['/Users/vallurileelasaikrishna/Documents/chess/results/models/rfc_61.18.joblib']